# Supervised Classification: Pilze

In diesem Notebook orientieren wir uns an der Struktur aus dem Unterrichtsnotebook `03_supervised_classification.ipynb`.

Ziel ist es vorherzusagen, ob ein Pilz essbar (`e`) oder giftig (`p`) ist.

In [ ]:
from io import StringIO
from pathlib import Path

import numpy as np
import pandas as pd
from pandas import read_csv

from sklearn.model_selection import train_test_split

from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

## Daten laden

Der Datensatz ist mit `~` getrennt. Beim normalen Einlesen gab es ein paar Zeilen mit leeren Zusatzfeldern am Ende. Deshalb bereinigen wir diese Zeilen direkt beim Laden.

In [ ]:
data_path = Path("../data/raw/mushroom_classification_data.csv")

cleaned_lines = []
fixed_lines = []

for line_number, line in enumerate(data_path.read_text(encoding="utf-8").splitlines(), start=1):
    parts = line.split("~")

    if len(parts) > 23 and all(value == "" for value in parts[23:]):
        fixed_lines.append(line_number)
        parts = parts[:23]

    cleaned_lines.append("~".join(parts))

data = read_csv(StringIO("\n".join(cleaned_lines)), sep="~")

print("Daten geladen:", data.shape)
print("Korrigierte Zeilen:", fixed_lines)
data.head()

## Daten kurz explorieren

Diesen Teil gibt es im Unterrichtsnotebook nur sehr kurz, aber für unsere Gruppenarbeit sollen wir den Datensatz auch anschauen und mögliche Probleme finden.

In [ ]:
print(data.shape)
print(data["poison"].value_counts())

In [ ]:
(data == "?").sum()[(data == "?").sum() > 0]

In [ ]:
data.nunique().sort_values()

Wir haben gesehen:

- Die Zielvariable ist relativ ausgeglichen.
- `stalk-root` enthält viele `?`-Werte.
- `veil-type` hat nur einen einzigen Wert und bringt deshalb wahrscheinlich nichts für das Modell.
- Ein paar Werte wirken kaputt, zum Beispiel `d**`, `0.34`, `0.3` und `0.4`.

In [ ]:
data = data.replace(["d**", "0.34", "0.3", "0.4"], "?")
data.to_csv("../data/processed/mushroom_classification_clean.csv", index=False)

## Daten Vertical splitten (Input/Output)

Wie im Unterricht trennen wir jetzt die Eingabedaten (`X`) von der Zielvariable (`y`).

In [ ]:
X = data.drop(columns=[
    "poison",
    "veil-type"
])

y = data[
    "poison"
]

## Daten horizontal Splitten (Train/Test)

Danach teilen wir die Daten in Trainingsdaten und Testdaten auf.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

## Daten Vorverarbeiten

Hier verwenden wir eine ähnliche Pipeline wie im Unterricht. Da unsere Daten nur kategoriale Spalten haben, brauchen wir keine Skalierung. Fehlende Werte (`?`) werden vorher zu `NaN`, dann ersetzt und anschließend mit One-Hot-Encoding umgewandelt.

In [ ]:
X_train = X_train.replace("?", np.nan)
X_test = X_test.replace("?", np.nan)

cat_selector = make_column_selector(dtype_exclude=["number"])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="constant", fill_value="missing")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(transformers=[
    ("cat", categorical_transformer, cat_selector)
])

Wir fitten die Vorverarbeitung nur auf den Trainingsdaten und wenden sie danach auf Trainings- und Testdaten an.

In [ ]:
preprocessor.fit(X_train)
X_train_prep = preprocessor.transform(X_train)
X_test_prep = preprocessor.transform(X_test)

## Model Entwicklung (Machine Learning)

Wie im Kursnotebook testen wir zuerst einen Entscheidungsbaum.

In [ ]:
tree_classifier = DecisionTreeClassifier(
    max_depth=5,
    min_samples_split=10,
    min_samples_leaf=3,
    random_state=42
)

tree_classifier.fit(X_train_prep, y_train)

tree_prediction = tree_classifier.predict(X_test_prep)

print("F1", f1_score(y_test, tree_prediction, pos_label="p"))
print("Recall", recall_score(y_test, tree_prediction, pos_label="p"))
print("Precision", precision_score(y_test, tree_prediction, pos_label="p"))
print("Accuracy", accuracy_score(y_test, tree_prediction))

Zur Kontrolle schauen wir auch kurz auf die Trainingsdaten. Wenn Training viel besser wäre als Test, könnte das auf Overfitting hindeuten.

In [ ]:
tree_train_prediction = tree_classifier.predict(X_train_prep)

print("F1", f1_score(y_train, tree_train_prediction, pos_label="p"))
print("Recall", recall_score(y_train, tree_train_prediction, pos_label="p"))
print("Precision", precision_score(y_train, tree_train_prediction, pos_label="p"))
print("Accuracy", accuracy_score(y_train, tree_train_prediction))

Zusätzlich testen wir noch KNN, weil das auch im Unterrichtsnotebook vorkommt.

In [ ]:
knn_classifier = KNeighborsClassifier(
    n_neighbors=9
)

knn_classifier.fit(X_train_prep, y_train)

knn_prediction = knn_classifier.predict(X_test_prep)

print("F1", f1_score(y_test, knn_prediction, pos_label="p"))
print("Recall", recall_score(y_test, knn_prediction, pos_label="p"))
print("Precision", precision_score(y_test, knn_prediction, pos_label="p"))
print("Accuracy", accuracy_score(y_test, knn_prediction))

## Fazit

Beide Modelle funktionieren auf diesem Datensatz sehr gut. Der Entscheidungsbaum ist für uns am besten geeignet, weil man ihn leichter erklären kann.

Wichtig ist bei diesem Thema vor allem der Recall für giftige Pilze. Ein giftiger Pilz, der als essbar vorhergesagt wird, wäre der gefährlichste Fehler.

## LLM-Nutzung

Wir haben ein LLM als Unterstützung benutzt, vor allem für Struktur, Formulierungen und zum Finden von Datenproblemen. Die Umsetzung orientiert sich aber bewusst am Kursnotebook und wurde einfach gehalten.